In [4]:
import json
import os
import torch
from datasets import Dataset, Audio
from transformers import VitsModel, VitsTokenizer, TrainingArguments, Trainer

model_id = "facebook/mms-tts-rus"
model = VitsModel.from_pretrained(model_id)
tokenizer = VitsTokenizer.from_pretrained(model_id)

def fine_tune(dataset_folder: str):
    with open(os.path.join(dataset_folder, "index.json"), "r", encoding='utf-8') as f:
        json_data = json.load(f)
    prepared_dataset = [
        {
            "file": filename,
            "text": json_data[filename]
        } for filename in json_data]

    dataset = Dataset.from_dict(
        {"file": [d["file"] for d in prepared_dataset], "text": [d["text"] for d in prepared_dataset]})
    dataset = dataset.cast_column("file", Audio(sampling_rate=16000))

    split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]

    # Freeze the text encoder (keeps phoneme understanding intact)
    for param in model.text_encoder.parameters():
        param.requires_grad = False

    # 2. FREEZE Flow and Decoder (allows learning speaker prosody and timbre)
    for param in model.flow.parameters():
        param.requires_grad = True
    for param in model.decoder.parameters():
        param.requires_grad = True


    tokenized_train = train_dataset.map(preprocess_function, batched=False, remove_columns=train_dataset.column_names)
    tokenized_eval = eval_dataset.map(preprocess_function, batched=False, remove_columns=eval_dataset.column_names)

    training_args = TrainingArguments(
        output_dir="./mms-tts-rus-finetuned-14m",
        per_device_train_batch_size=1,       # Keep at 1 to save VRAM
        gradient_accumulation_steps=4,       # Effective batch size = 4
        learning_rate=2.5e-5,                # Increased from 1e-5 (sweet spot for this data size)
        weight_decay=0.01,
        max_steps=3000,                      # Increased significantly (approx 30-50 epochs depending on batch size)
        eval_steps=500,                      # Evaluate every 500 steps
        save_steps=500,                      # Save checkpoint every 500 steps
        eval_strategy="steps",         # Enable validation monitoring
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        load_best_model_at_end=True,         # Automatically rolls back to the checkpoint with the lowest eval loss
        metric_for_best_model="eval_loss",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
    )

    trainer.train()
    trainer.save_model("./mms-rus-finetuned-final")

def preprocess_function(examples):
    # Tokenize text
    text_inputs = tokenizer(examples["text"], padding="max_length", max_length=512, truncation=True)

    return {
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
        "labels": examples["file"]["array"],
    }


fine_tune("./sonya_dataset")

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

NotImplementedError: Training of VITS is not supported yet.